# 3. OpenFDA Inference Pipeline

Apply trained model to OpenFDA NDC drugs data with web-enhanced LLM validation.


In [1]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/1-30-26')

import os
import re
import pandas as pd
import numpy as np
import json
from pathlib import Path

from splink import Linker, DuckDBAPI

from config import config
from utils import DatabaseManager, log_step, Timer, save_checkpoint, load_checkpoint, load_json
from data_prep import (
    load_dim_org, create_unified_schema, filter_bad_records, create_blocking_keys,
    add_distinctive_tokens, add_token_set_features, compute_token_statistics,
    get_corpus_stopwords, rollup_to_parent
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print("Imports loaded")


Imports loaded


In [2]:
# Initialize database and load model
db = DatabaseManager()

model_path = config.paths.MODEL_FILE
if not Path(model_path).exists():
    raise FileNotFoundError(f"Model not found at {model_path}. Run 2_training.ipynb first.")

model_json = load_json(model_path, "Trained model")

print(f"\nMODEL LOADED")
print("=" * 50)
print(f"Path: {model_path}")
if 'training_metadata' in model_json:
    meta = model_json['training_metadata']
    print(f"Trained on: {meta.get('timestamp', 'N/A')}")
    print(f"Training records: {meta.get('dim_org_records', 0):,} dim_org + {meta.get('grid_records', 0):,} GRID")


[16:19:36]  Loading JSON: Trained model

MODEL LOADED
Path: /Users/robertlalani/Desktop/entity_resolution_12_18_25/1-30-26/models/model_v1.json
Trained on: 2026-02-02T16:19:21.421583
Training records: 100,253 dim_org + 102,173 GRID


In [3]:
# Load reference data (dim_org)
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")

if cached_dim_org is not None:
    dim_org_df = cached_dim_org
    print(f"Loaded cached dim_org: {len(dim_org_df):,} records")
else:
    dim_org_raw = load_dim_org(db, sample_size=None)
    dim_org_df = create_unified_schema(dim_org_raw, 'dim_org')
    dim_org_df, _ = filter_bad_records(dim_org_df)
    dim_org_df = create_blocking_keys(dim_org_df)
    print(f"Loaded dim_org: {len(dim_org_df):,} records")


[16:19:38]  Loading checkpoint: dim_org cache
[16:19:38]    Loaded 109,851 rows
Loaded cached dim_org: 109,851 records


In [4]:
# Query top N most frequent OpenFDA organizations
TOP_N_ORGS = 10000

top_orgs_query = f"""
    SELECT name, COUNT(*) as frequency
    FROM allsci_prod_gold.potential_mismatched_organizations
    WHERE source_table = 'open_fda_silver.ndc_drugs'
    GROUP BY name
    ORDER BY frequency DESC
    LIMIT {TOP_N_ORGS}
"""

top_orgs_df = db.execute_query(top_orgs_query, f"Top {TOP_N_ORGS} OpenFDA organizations")

print(f"\nTOP {TOP_N_ORGS} MOST FREQUENT OPENFDA ORGANIZATIONS")
print("=" * 60)
print(f"Total unique orgs: {len(top_orgs_df):,}")
print(f"Total records covered: {top_orgs_df['frequency'].sum():,}")
print(f"\nTop 10 by frequency:")
print(top_orgs_df.head(10).to_string())


[16:19:51]  Executing: Top 10000 OpenFDA organizations
[16:19:55]    Returned 10,000 rows in 4.0s

TOP 10000 MOST FREQUENT OPENFDA ORGANIZATIONS
Total unique orgs: 10,000
Total records covered: 301,600

Top 10 by frequency:
                                         name  frequency
0                        Bryant Ranch Prepack       9504
1                Hahnemann Laboratories, INC.       8799
2                    A-S Medication Solutions       5850
3                           REMEDYREPACK INC.       5339
4                                      Boiron       4213
5                            Proficient Rx LP       3680
6                    Aurobindo Pharma Limited       3063
7                    Greer Laboratories, Inc.       2654
8                 PD-Rx Pharmaceuticals, Inc.       2460
9  The Procter & Gamble Manufacturing Company       2280


In [5]:
# Compute IDF scores from reference data
idf_scores = compute_token_statistics(dim_org_df, 'name_normalized')
corpus_stopwords = get_corpus_stopwords(idf_scores, percentile=0.10)

print(f"\nIDF SCORES COMPUTED")
print("=" * 50)
print(f"Vocabulary size: {len(idf_scores):,} tokens")
print(f"Corpus stopwords: {len(corpus_stopwords):,}")


[16:19:59]  Computing token IDF statistics...
[16:19:59]    Computed IDF for 64,820 unique tokens
[16:19:59]    IDF range: 1.90 (most common) to 11.61 (most rare)
[16:19:59]    Auto-identified 6786 corpus stopwords (IDF <= 10.00)

IDF SCORES COMPUTED
Vocabulary size: 64,820 tokens
Corpus stopwords: 6,786


In [6]:
# Load one record per org for top N organizations
filtered_query = f"""
WITH top_orgs AS (
    SELECT name, COUNT(*) as frequency
    FROM allsci_prod_gold.potential_mismatched_organizations
    WHERE source_table = 'open_fda_silver.ndc_drugs'
    GROUP BY name
    ORDER BY frequency DESC
    LIMIT {TOP_N_ORGS}
),
one_per_org AS (
    SELECT m.*, 
           ROW_NUMBER() OVER (PARTITION BY m.name ORDER BY m.mismatch_id) as rn
    FROM allsci_prod_gold.potential_mismatched_organizations m
    INNER JOIN top_orgs t ON m.name = t.name
    WHERE m.source_table = 'open_fda_silver.ndc_drugs'
)
SELECT * FROM one_per_org WHERE rn = 1
"""

log_step(f"Loading top {TOP_N_ORGS:,} organizations from OpenFDA...")
openfda_df = db.execute_query(filtered_query, "OpenFDA top organizations")
if 'rn' in openfda_df.columns:
    openfda_df = openfda_df.drop(columns=['rn'])

print(f"\nLOADED: {len(openfda_df):,} records")


[16:20:01]  Loading top 10,000 organizations from OpenFDA...
[16:20:01]  Executing: OpenFDA top organizations
[16:20:14]    Returned 10,000 rows in 13.4s

LOADED: 10,000 records


In [7]:
# Parse metadata
def parse_metadata_string(metadata_str):
    if pd.isna(metadata_str) or not metadata_str:
        return {}
    content = str(metadata_str).strip('{}')
    result = {}
    current_key = None
    current_value = []
    for part in re.split(r',\s*(?=[a-z_]+=)', content):
        if '=' in part:
            if current_key:
                result[current_key] = ''.join(current_value).strip()
            key_val = part.split('=', 1)
            current_key = key_val[0].strip()
            current_value = [key_val[1]] if len(key_val) > 1 else []
        else:
            current_value.append(', ' + part)
    if current_key:
        result[current_key] = ''.join(current_value).strip()
    return result

print("Parsing metadata...")
metadata_parsed = openfda_df['metadata'].apply(parse_metadata_string)
metadata_df = pd.DataFrame(metadata_parsed.tolist())
openfda_df = pd.concat([openfda_df.drop(columns=['metadata']), metadata_df], axis=1)
print(f"Parsed columns: {list(openfda_df.columns)}")


Parsing metadata...
Parsed columns: ['mismatch_id', 'name', 'source_table', 'source_entity_id', 'ingestion_datetime', 'product_ndc', 'spl_id', 'name_clean', 'relationship_type', 'drug_generic_name', 'name_normalized', 'drug_brand_name']


In [8]:
# Prepare data for Splink
source_id_mapping_raw = openfda_df[['mismatch_id', 'source_entity_id']].copy()

missing_cols = ['country', 'country_code', 'city', 'state', 'latitude', 'longitude', 'name_aliases', 'type', 'name_prefix_5', 'name_prefix_10']
for col in missing_cols:
    if col not in openfda_df.columns:
        openfda_df[col] = None

chunk_unified = create_unified_schema(openfda_df, 'mismatched')
chunk_unified['mismatch_id'] = chunk_unified['unique_id'].str.replace('mis_', '', regex=False)
chunk_unified = chunk_unified.merge(source_id_mapping_raw, on='mismatch_id', how='left', suffixes=('_drop', ''))
drop_cols = [c for c in chunk_unified.columns if c.endswith('_drop')]
chunk_unified = chunk_unified.drop(columns=drop_cols, errors='ignore')

source_id_mapping = chunk_unified[['unique_id', 'source_entity_id']].copy()
print(f"source_entity_id mapped: {source_id_mapping['source_entity_id'].notna().sum():,} records")

chunk_filtered, removed = filter_bad_records(chunk_unified)
print(f"Filtered: {len(removed):,} bad records")

chunk_prepared = create_blocking_keys(chunk_filtered)
chunk_prepared = add_distinctive_tokens(chunk_prepared, idf_scores, corpus_stopwords)
chunk_prepared = add_token_set_features(chunk_prepared)

print(f"\nPREPARED: {len(chunk_prepared):,} records ready for inference")


[16:20:48]  Creating unified schema for mismatched...
[16:20:48]    Created unified schema: 10,000 rows, 18 columns
source_entity_id mapped: 10,000 records
[16:20:48]  Filtering bad records...
[16:20:48]    Removed 10 of 10,000 records (0.1%)
[16:20:48]      - short_name: 10
Filtered: 10 bad records
[16:20:48]  Creating blocking keys...
[16:20:48]  Adding phonetic codes...
[16:20:48]    Phonetic coverage: 100.0%
[16:20:48]    Added blocking keys to 9,990 records
[16:20:48]  Adding distinctive tokens...
[16:20:49]    Loaded 34,655 geographic stopwords
[16:20:49]    Geographic stopwords: 34,655 (locations filtered out)
[16:20:49]    Distinctive token coverage: 100.0%
[16:20:49]  Adding token set features for containment detection...
[16:20:49]    Average token count: 2.6

PREPARED: 9,990 records ready for inference


In [9]:
# Ensure dim_org has distinctive token features
if 'distinctive_token' not in dim_org_df.columns:
    print("Adding distinctive tokens to dim_org...")
    dim_org_df = add_distinctive_tokens(dim_org_df, idf_scores, corpus_stopwords)
    dim_org_df = add_token_set_features(dim_org_df)


Adding distinctive tokens to dim_org...
[16:20:53]  Adding distinctive tokens...
[16:20:53]    Geographic stopwords: 34,655 (locations filtered out)
[16:20:53]    Distinctive token coverage: 99.9%
[16:20:53]  Adding token set features for containment detection...
[16:20:54]    Average token count: 3.2


In [ ]:
# Run inference with geo-agnostic blocking rules
log_step("Running Splink inference...")

model_settings = {k: v for k, v in model_json.items() if k != 'training_metadata'}

# Override blocking rules for geo-sparse inference data
inference_blocking_rules = [
    {"blocking_rule": "l.name_normalized = r.name_normalized", "sql_dialect": "duckdb"},
    {"blocking_rule": "l.distinctive_token = r.distinctive_token", "sql_dialect": "duckdb"},
    {"blocking_rule": "l.name_metaphone = r.name_metaphone", "sql_dialect": "duckdb"},
    {"blocking_rule": "substr(l.name_normalized, 1, 10) = substr(r.name_normalized, 1, 10)", "sql_dialect": "duckdb"},
]
model_settings['blocking_rules_to_generate_predictions'] = inference_blocking_rules
print(f"Using {len(inference_blocking_rules)} inference blocking rules (geo-agnostic)")

linker = Linker([dim_org_df, chunk_prepared], model_settings, db_api=DuckDBAPI())

predictions = linker.inference.predict(threshold_match_probability=config.matching.THRESHOLD_PREDICTION)
predictions_df = predictions.as_pandas_dataframe()
predictions_df['source_table'] = 'open_fda_silver.ndc_drugs'

print(f"\nINFERENCE COMPLETE: {len(predictions_df):,} predictions")


In [ ]:
# Add source_entity_id to predictions
predictions_df = predictions_df.merge(
    source_id_mapping,
    left_on='unique_id_r',
    right_on='unique_id',
    how='left',
    suffixes=('', '_map')
)

print(f"Predictions with source_entity_id: {predictions_df['source_entity_id'].notna().sum():,}")

# Score distribution
print("\nSCORE DISTRIBUTION:")
bins = [0, 0.5, 0.7, 0.85, 0.95, 1.0]
labels = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-0.95', '0.95-1.0']
predictions_df['score_bin'] = pd.cut(predictions_df['match_probability'], bins=bins, labels=labels)
print(predictions_df['score_bin'].value_counts().sort_index())


In [ ]:
# Save predictions before LLM validation
save_checkpoint(predictions_df, config.paths.DATA_DIR + "/openfda_predictions.parquet", "Predictions")
predictions_df.to_csv(config.paths.DATA_DIR + "/openfda_predictions.csv", index=False)
print(f"Saved {len(predictions_df):,} predictions")


## Web-Enhanced LLM Validation

Use FDA labels and web search to validate matches with Claude.


In [ ]:
# Configuration for LLM validation
USE_WEB_ENHANCED_VALIDATION = True
config.llm_judge.ENABLE_LLM_VALIDATION = True
MAX_WORKERS = 10
NUM_TO_PROCESS = 1000  # Set to number of predictions to validate

if USE_WEB_ENHANCED_VALIDATION:
    from anthropic import Anthropic
    from web_search import get_match_evidence
    from llm_judge import judge_match_with_evidence
    
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')
    SERPER_API_KEY = os.environ.get('SERPER_API_KEY')
    
    if not ANTHROPIC_API_KEY:
        print("WARNING: ANTHROPIC_API_KEY not set, LLM validation disabled")
        USE_WEB_ENHANCED_VALIDATION = False
    elif not SERPER_API_KEY:
        print("WARNING: SERPER_API_KEY not set, web search disabled but LLM validation will work")
    else:
        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        print(f"LLM validation ready: {config.llm_judge.PRIMARY_MODEL}")
        print(f"Will process: {NUM_TO_PROCESS:,} predictions with {MAX_WORKERS} workers")


In [ ]:
def process_single(row, dim_org_lookup, client):
    """Process a single prediction with web evidence and LLM validation."""
    try:
        ndc_code = None
        source_entity_id = row.get('source_entity_id', '')
        if source_entity_id and '_' in str(source_entity_id):
            parts = str(source_entity_id).split('_')
            if len(parts) >= 1:
                ndc_code = parts[0]
        
        source_org = row.get('name_r', '')
        matched_id = row.get('unique_id_l', '')
        matched_info = dim_org_lookup.get(matched_id, {})
        matched_org = matched_info.get('name', row.get('name_l', ''))
        
        evidence = get_match_evidence(
            source_org=source_org,
            matched_org=matched_org,
            ndc_code=ndc_code
        )
        
        llm_result = judge_match_with_evidence(
            row=row,
            dim_org_record=matched_info,
            evidence=evidence,
            client=client,
            model=config.llm_judge.PRIMARY_MODEL
        )
        
        return {
            'unique_id_l': row['unique_id_l'],
            'unique_id_r': row['unique_id_r'],
            'match_probability': row['match_probability'],
            'llm_match': llm_result.get('match'),
            'llm_confidence': llm_result.get('confidence'),
            'llm_reason': llm_result.get('reason'),
            'fda_labeler': evidence.get('fda_labeler'),
            'fda_drug': evidence.get('fda_drug'),
            'source_org': source_org,
            'matched_org': matched_org,
        }
    except Exception as e:
        return {
            'unique_id_l': row.get('unique_id_l'),
            'unique_id_r': row.get('unique_id_r'),
            'match_probability': row.get('match_probability'),
            'llm_match': None,
            'llm_confidence': 0.0,
            'llm_reason': f'error: {str(e)[:100]}',
            'fda_labeler': None,
            'fda_drug': None,
            'source_org': row.get('name_r', ''),
            'matched_org': row.get('name_l', ''),
        }

print("process_single defined")


In [ ]:
if USE_WEB_ENHANCED_VALIDATION:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm
    
    # Build dim_org lookup
    dim_org_lookup = dim_org_df.set_index('unique_id').to_dict('index')
    
    # Select predictions to validate
    to_validate = predictions_df.head(NUM_TO_PROCESS).to_dict('records')
    print(f"Validating {len(to_validate):,} predictions...")
    
    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_single, row, dim_org_lookup, client): i
            for i, row in enumerate(to_validate)
        }
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="LLM Validation"):
            result = future.result()
            results.append(result)
    
    validation_df = pd.DataFrame(results)
    print(f"\nValidation complete: {len(validation_df):,} results")
    
    # Summary
    if 'llm_match' in validation_df.columns:
        print("\nLLM VALIDATION SUMMARY:")
        print(validation_df['llm_match'].value_counts(dropna=False))
else:
    print("LLM validation skipped")


In [ ]:
# Save validation results
if USE_WEB_ENHANCED_VALIDATION and len(results) > 0:
    save_checkpoint(validation_df, config.paths.DATA_DIR + "/openfda_llm_validated.parquet", "LLM validated")
    
    # Separate confirmed vs rejected
    confirmed = validation_df[validation_df['llm_match'] == True]
    rejected = validation_df[validation_df['llm_match'] == False]
    
    save_checkpoint(confirmed, config.paths.DATA_DIR + "/openfda_llm_confirmed.parquet", "LLM confirmed")
    save_checkpoint(rejected, config.paths.DATA_DIR + "/openfda_llm_rejected.parquet", "LLM rejected")
    
    print(f"\nFINAL RESULTS:")
    print(f"  LLM Confirmed: {len(confirmed):,}")
    print(f"  LLM Rejected:  {len(rejected):,}")
    print(f"  Errors/Null:   {len(validation_df) - len(confirmed) - len(rejected):,}")


In [ ]:
# Clean up
db.close()
print("\n" + "=" * 50)
print("INFERENCE COMPLETE")
print("=" * 50)
